In [1]:
import os


In [2]:
%pwd

'c:\\Users\\nalin\\TinyTales\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\nalin\\TinyTales'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir:Path
    data_path:Path
    tokenizer_name:str

In [6]:
from Tinytales.constants import *
from Tinytales.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config


In [8]:
import os
from Tinytales.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk


c:\Users\nalin\TinyTales\venv31\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class DataTransformation:

    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(
            config.tokenizer_name
        )

    def convert_examples_to_features(self, batch):

        input_encodings = self.tokenizer(
            batch['dialogue'],
            max_length=1024,
            truncation=True
        )

        with self.tokenizer.as_target_tokenizer():
            target_encodings = self.tokenizer(
                batch['summary'],
                max_length=128,
                truncation=True
            )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']

        }
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))

In [11]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-10-26 16:05:23,918: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-26 16:05:23,921: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-26 16:05:23,922: INFO: common: created directory at: artifacts]
[2026-10-26 16:05:23,923: INFO: common: created directory at: artifacts/data_transformation]
[2026-10-26 16:05:24,362: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail "HTTP/1.1 200 OK"]
[2026-10-26 16:05:24,668: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-10-26 16:05:24,975: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"]


Map:   0%|          | 0/14732 [00:00<?, ? examples/s]


AttributeError: PegasusTokenizer has no attribute as_target_tokenizer